In [1]:
import pandas as pd
import numpy as np

In [2]:
caminho = r"C:\Users\Fábio Padilha\Aulas_Ada_Turma_1659\analise_titanic\titanic_dataset.csv"

In [3]:
df = pd.read_csv(caminho)
print(f"✅ Dataset carregado: {df.shape[0]} linhas × {df.shape[1]} colunas")
print("\n📊 Primeiras 5 linhas:")
print(df.head())

✅ Dataset carregado: 891 linhas × 12 colunas

📊 Primeiras 5 linhas:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C12

In [4]:
print("📋 1. DESCRIÇÃO DOS DADOS")
print("=" * 60)

# 1.1 Informações gerais
print("\n1.1 INFO GERAL:")
print(df.info())

# 1.2 Valores NULOS
print("\n1.2 NULOS POR COLUNA:")
nulos = pd.DataFrame({
    'Coluna': df.columns,
    'Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df) * 100).round(2)
})
print(nulos)

# 1.3 Valores distintos
print("\n1.3 VALORES DISTINTOS:")
distintos = pd.DataFrame({
    'Coluna': df.columns,
    'Distintos': [df[col].nunique() for col in df.columns],
    'Tipo': df.dtypes
})
print(distintos)

# 1.4 Estatísticas descritivas
print("\n1.4 ESTATÍSTICAS DESCRITIVAS:")
print(df.describe(include='all').round(2))

📋 1. DESCRIÇÃO DOS DADOS

1.1 INFO GERAL:
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB
None

1.2 NULOS POR COLUNA:
                  Coluna  Nulos  % Nulos
PassengerId  PassengerId      0     0.00
Survived        Survived      0     0.00
Pclass            Pclass      0     0.00
Name                Name

In [5]:
print("🔧 2. TRATAMENTO DE DADOS")
print("\nNulos ANTES:", df.isnull().sum().sum())

# 2.1 Tratamento NULOS - Campo por campo
# Age: mediana por Pclass e Sex
df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))

# Cabin: 77% nulos → remover
df = df.drop('Cabin', axis=1)

# Embarked: moda
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# 2.2 Duplicatas
print(f"\nDuplicatas ANTES: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Duplicatas DEPOIS: {df.duplicated().sum()}")

print("\n✅ Nulos DEPOIS:", df.isnull().sum().sum())
print(f"Shape final: {df.shape}")

🔧 2. TRATAMENTO DE DADOS

Nulos ANTES: 866

Duplicatas ANTES: 0
Duplicatas DEPOIS: 0

✅ Nulos DEPOIS: 0
Shape final: (891, 11)


In [6]:
# 2.3 Variáveis derivadas
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# AgeGroup
def categorizar_idade(idade):
    if idade < 13: return 'Criança'
    elif idade < 18: return 'Adolescente'
    elif idade < 60: return 'Adulto'
    else: return 'Idoso'

df['AgeGroup'] = df['Age'].apply(categorizar_idade)

print("✅ VARIÁVEIS DERIVADAS CRIADAS:")
print(df[['FamilySize', 'IsAlone', 'AgeGroup']].head(10))
print(f"\nTipos corrigidos:")
print(df.dtypes)

✅ VARIÁVEIS DERIVADAS CRIADAS:
   FamilySize  IsAlone     AgeGroup
0           2        0       Adulto
1           2        0       Adulto
2           1        1       Adulto
3           2        0       Adulto
4           1        1       Adulto
5           1        1       Adulto
6           1        1       Adulto
7           5        0      Criança
8           3        0       Adulto
9           2        0  Adolescente

Tipos corrigidos:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Embarked           str
FamilySize       int64
IsAlone          int64
AgeGroup           str
dtype: object


In [7]:
print("🎯 3. PERGUNTAS")
print("=" * 60)

# PERGUNTA 1: Taxa sobrevivência por Classe × Gênero
print("\n📊 P1: SOBREVIVÊNCIA POR CLASSE × GÊNERO")
q1 = df.groupby(['Pclass', 'Sex'])['Survived'].agg(
    Total='count', Sobreviventes='sum', Taxa='mean'
).round(4)
q1['Taxa_%'] = (q1['Taxa'] * 100).round(1)
print(q1.sort_index())

🎯 3. PERGUNTAS

📊 P1: SOBREVIVÊNCIA POR CLASSE × GÊNERO
               Total  Sobreviventes    Taxa  Taxa_%
Pclass Sex                                         
1      female     94             91  0.9681    96.8
       male      122             45  0.3689    36.9
2      female     76             70  0.9211    92.1
       male      108             17  0.1574    15.7
3      female    144             72  0.5000    50.0
       male      347             47  0.1354    13.5


In [8]:
# PERGUNTA 2: Faixa etária com maior sobrevivência
print("\n📊 P2: SOBREVIVÊNCIA POR FAIXA ETÁRIA")
q2 = df.groupby('AgeGroup')['Survived'].agg(
    Total='count', Taxa='mean'
).round(4)
q2['Taxa_%'] = (q2['Taxa'] * 100).round(1)
print(q2.sort_values('Taxa', ascending=False))


📊 P2: SOBREVIVÊNCIA POR FAIXA ETÁRIA
             Total    Taxa  Taxa_%
AgeGroup                          
Criança         69  0.5797    58.0
Adolescente     44  0.4773    47.7
Adulto         752  0.3644    36.4
Idoso           26  0.2692    26.9


In [9]:
# PERGUNTA 3: Sozinho vs Acompanhado
print("\n📊 P3: SOZINHO vs ACOMPANHADO")
q3 = df.groupby('IsAlone')['Survived'].agg(
    Total='count', Taxa='mean'
).round(4)
q3['Status'] = ['Acompanhado', 'Sozinho']
q3['Taxa_%'] = (q3['Taxa'] * 100).round(1)
print(q3[['Status', 'Total', 'Taxa_%']])


📊 P3: SOZINHO vs ACOMPANHADO
              Status  Total  Taxa_%
IsAlone                            
0        Acompanhado    354    50.6
1            Sozinho    537    30.4


In [10]:
# PERGUNTA 4: Porto de embarque
print("\n📊 P4: SOBREVIVÊNCIA POR PORTO")
q4 = df.groupby('Embarked')['Survived'].agg(
    Total='count', Taxa='mean'
).round(4)
q4['Taxa_%'] = (q4['Taxa'] * 100).round(1)
print(q4.sort_values('Taxa', ascending=False))


📊 P4: SOBREVIVÊNCIA POR PORTO
          Total    Taxa  Taxa_%
Embarked                       
C           168  0.5536    55.4
Q            77  0.3896    39.0
S           646  0.3390    33.9


In [11]:
# PERGUNTA 5: Perfil do sobrevivente típico
print("\n👤 P5: PERFIL DO SOBREVIVENTE TÍPICO")
sobreviventes = df[df['Survived'] == 1]
perfil = pd.DataFrame({
    'Métrica': ['Classe_Média', 'Sexo_Maior', 'Idade_Média', 'Tarifa_Média', 'FamilySize_Média'],
    'Valor': [
        sobreviventes['Pclass'].mean(),
        sobreviventes['Sex'].mode()[0],
        sobreviventes['Age'].mean(),
        sobreviventes['Fare'].mean(),
        sobreviventes['FamilySize'].mean()
    ]
}).round(2)
print(perfil)


👤 P5: PERFIL DO SOBREVIVENTE TÍPICO
            Métrica      Valor
0      Classe_Média   1.950292
1        Sexo_Maior     female
2       Idade_Média  28.108684
3      Tarifa_Média  48.395408
4  FamilySize_Média   1.938596


In [12]:
# Dataset final limpo
cols_finais = ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 
               'Embarked', 'FamilySize', 'IsAlone', 'AgeGroup']
df_final = df[cols_finais].copy()

# Exportar
df_final.to_csv(r'C:\Users\Fábio Padilha\Aulas_Ada_Turma_1659\analise_titanic\titanic_limpo.csv', index=False)

print("🏆 RESUMO")
print("=" * 60)
print(f"📊 Dataset original: {df.shape}")
print(f"📊 Dataset limpo: {df_final.shape}")
print(f"✅ Nulos restantes: 0")
print(f"✅ Duplicatas: 0")
print(f"📈 Taxa sobrevivência geral: {df['Survived'].mean():.1%}")
print(f"\n💾 Arquivos salvos:")
print(f"   • titanic_limpo.csv")
print("\n✅ CONCLUÍDO!")

🏆 RESUMO
📊 Dataset original: (891, 14)
📊 Dataset limpo: (891, 11)
✅ Nulos restantes: 0
✅ Duplicatas: 0
📈 Taxa sobrevivência geral: 38.4%

💾 Arquivos salvos:
   • titanic_limpo.csv

✅ CONCLUÍDO!
